# From lab notes to FEM material card in one template

You have run a tensile test on a batch of 316L stainless steel.
You have the numbers: yield strength, tensile strength, elongation.
You also have a constitutive model fit (Hockett-Sherby) and you want
a material card for your FEM solver.

This notebook does all of that. Here is the plan:

1. You fill in **one template**, and the code does the rest.
2. You run the cells below it.
3. You get:
   - A structured, connected record of everything: production, test, calibration, material card
   - Answers to questions like *who ran the test?* and *which machine was used?*
   - Ready-to-use solver files for LS-Dyna and Abaqus

No modelling decisions, no data format choices, no manual copy-pasting.
The template pre-decides all of that.

---

> **Acknowledgement.** This workflow is inspired by work done in the
> [StahlDigital project](https://www.iwm.fraunhofer.de/de/ueber-uns/loesungen-fuer-produktlebenzyklus/digitalisierung-in-der-werkstofftechnik/stahldigital.html)
> (Fraunhofer IWM). A detailed description of the underlying approach is
> published in:
>
> Roters, F.; Aslam, A.; Bai, Y.; Büschelberger, M.; Bulert, K.; Butz, A.;
> Hickel, T.; Jogi, T.; Klitschke, S.; Martin, M.; Meyer, L.-P.; Morand, L.;
> Nahshon, Y.; Radtke, N.; Saikia, U.; Trondl, A.; Wessel, A.; Zierep, P.;
> Helm, D. (2025).
> *StahlDigital: Ontology-Based Workflows for the Steel Industry.*
> Advanced Engineering Materials, 27: 2402148.
> <https://doi.org/10.1002/adem.202402148>

## Before you start

If you are running this locally, clone the repository and install the packages:

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
python3 -m venv .venv && source .venv/bin/activate
pip install semantic-schemas semantic-transformers jupyterlab
jupyter lab
```

In [1]:
%pip install -q semantic-schemas semantic-transformers

Note: you may need to restart the kernel to use updated packages.


## Step 1: Fill in the template

**This is the only cell you need to edit.**

Replace the example values with your own data.
The comments explain what each field means.
When you are done, run all the cells below, in order.

In [2]:
workflow_template = {

    # A name for this whole workflow run
    "workflow_name": "316L material card workflow, batch 1",

    # A web address (IRI) that uniquely identifies the person who ran the test.
    # If your organisation uses a people directory, use that URL.
    # Otherwise, make up a consistent address for each person.
    "operator_iri": "https://example.org/people/jane-doe",

    # A web address that uniquely identifies the test specimen.
    # Use the same address every time you refer to this specimen.
    "specimen_iri": "https://example.org/specimens/316L-batch-1-bar-1",

    # A web address that uniquely identifies the material batch.
    "material_iri": "https://example.org/materials/316L-batch-1",


    # ── Your testing machine ────────────────────────────────────────────────
    "device": {
        "name":             "Zwick Z250 Universal Testing Machine",
        "manufacturer":     "Zwick Roell",
        "model":            "Z250",
        "serial_number":    "ZR-12345",
        "calibration_date": "2024-03-01"   # ISO date: YYYY-MM-DD
    },


    # ── How the material was produced ───────────────────────────────────────
    "production": {
        "name":        "316L Production, batch 1",
        "description": "Melting, continuous casting, and hot rolling of 316L "
                       "austenitic stainless steel.",
        "conditions": [
            {"name": "Rolling Temperature", "value": 1150, "unit": "C"},
            {"name": "Final Thickness",     "value": 6.0,  "unit": "mm"}
        ]
    },


    # ── Tensile test conditions and results ─────────────────────────────────
    # The operator, instrument, and specimen above are automatically linked here.
    "characterization": {
        "name": "Tensile test 316L batch 1 QA",
        "date": "2024-11-15T09:30:00",     # ISO datetime: YYYY-MM-DDTHH:MM:SS
        "conditions": [
            {"name": "Test Standard", "unit": "ISO 6892-1"},
            {"name": "Strain Rate",   "value": 0.00025, "unit": "1/s"},
            {"name": "Temperature",   "value": 23,      "unit": "C"}
        ],
        # Measured property values — add or remove rows as needed
        # Allowed properties: YieldStrength, TensileStrength, UpperYieldStrength,
        #   LowerYieldStrength, ProofStrength, PercentageElongationAfterFracture,
        #   PercentagePermanentElongation, PercentageReductionOfArea
        "results": [
            {"property": "YieldStrength",                     "value": 285, "unit": "MPa"},
            {"property": "TensileStrength",                   "value": 620, "unit": "MPa"},
            {"property": "PercentageElongationAfterFracture", "value": 50,  "unit": "%"},
            {"property": "PercentageReductionOfArea",         "value": 65,  "unit": "%"}
        ]
    },


    # ── Constitutive model fit ──────────────────────────────────────────────
    # Allowed model types: Hockett-Sherby, Swift, Voce, Hollomon, Johnson-Cook
    "calibration": {
        "name":        "Hockett-Sherby calibration 316L batch 1",
        "model_type":  "Hockett-Sherby",
        "description": "Levenberg-Marquardt fit to ISO 6892-1 tensile test data.",
        "optimizer_settings": [
            {"name": "Optimiser",    "unit": "Levenberg-Marquardt"},
            {"name": "Strain Range", "value": 0.5, "unit": "1"}
        ],
        "parameters": [
            {"name": "sigma_sat", "value": 780.0, "unit": "MPa"},
            {"name": "sigma_0",   "value": 220.0, "unit": "MPa"},
            {"name": "c",         "value": 12.5},
            {"name": "n",         "value": 0.68}
        ]
    },


    # ── Elastic constants for the material card ─────────────────────────────
    # Plastic parameters are taken automatically from the calibration above.
    "material_card": {
        "name":        "316L stainless steel, Hockett-Sherby v1",
        "description": "Mechanical material card for 316L austenitic SS batch 1.",
        "density":        {"value": 7.93,  "unit": "g/cm3"},
        "youngs_modulus": {"value": 193.0, "unit": "GPa"},
        "poissons_ratio": {"value": 0.29}
    }

}

## Setup (run once, no need to read)

This cell loads the tools and defines the helper functions used in the steps below.
You do not need to understand what is inside. Just run it.

In [3]:
import json, math, pathlib, rdflib
from semantic_schemas import Schema

HERE    = pathlib.Path().resolve()
SCHEMAS = HERE.parents[4]                       # .../schemas/
WF_DIR  = HERE.parents[3] / "PMDCo"             # .../workflow/PMDCo/

_device_schema = Schema(SCHEMAS / "measurement-device" / "PMDCo")
_manuf_schema  = Schema(SCHEMAS / "manufacturing" / "step" / "base" / "PMDCo")
_proc_schema   = Schema(SCHEMAS / "characterization" / "process" / "PMDCo")
_tto_schema    = Schema(SCHEMAS / "characterization" / "step" / "tensile-test" / "TTO")
_calib_schema  = Schema(SCHEMAS / "simulation" / "step" / "model-calibration" / "PMDCo")
_card_schema   = Schema(SCHEMAS / "material-card" / "mechanical" / "PMDCo")
_wf_schema     = Schema(WF_DIR)

BASE = "https://example.org/"


def register_device(t):
    d = t["device"]
    oold = _device_schema.transform({
        "device_name": d["name"], "manufacturer": d["manufacturer"],
        "model": d["model"], "serial_number": d["serial_number"],
        "calibration_date": d["calibration_date"]
    })
    iri = BASE + oold["id"]
    graph = _device_schema.parse(oold, base=BASE)
    print(f"Instrument registered : {d['name']}")
    print(f"  Model / S/N         : {d['model']} / {d['serial_number']}")
    print(f"  Last calibration    : {d['calibration_date']}")
    print(f"  Identifier          : {iri}")
    return iri, graph


def record_production(t):
    p = t["production"]
    oold = _manuf_schema.transform({
        "process_name": p["name"],
        "description":  p.get("description", ""),
        "outputs":      [t["material_iri"]],
        "conditions":   p.get("conditions", [])
    })
    iri = oold["id"]
    graph = _manuf_schema.parse(oold, base=BASE)
    print(f"Production step recorded : {p['name']}")
    print(f"  Output material      : {t['material_iri']}")
    for c in p.get("conditions", []):
        val = f"{c['value']} " if "value" in c else ""
        print(f"  {c['name']:<22}: {val}{c.get('unit', '')}")
    return iri, graph


def record_characterization(t, device_iri):
    c = t["characterization"]
    proc_oold = _proc_schema.transform({
        "process_name": c["name"],
        "operator_iri": t["operator_iri"],
        "device_iri":   device_iri,
        "specimen_iri": t["specimen_iri"],
        "date":         c.get("date"),
        "conditions":   c.get("conditions", [])
    })
    tto_oold = _tto_schema.transform({
        "test_name":    c["name"],
        "specimen_iri": t["specimen_iri"],
        "conditions":   c.get("conditions", []),
        "results":      [{"property": r["property"], "value": r["value"], "unit": r["unit"]}
                         for r in c["results"]]
    })
    proc_oold["step_reference"] = BASE + tto_oold["id"]
    proc_graph = _proc_schema.parse(proc_oold, base=BASE)
    tto_graph  = _tto_schema.parse(tto_oold,  base=BASE)
    print(f"Tensile test recorded : {c['name']}")
    print(f"  Operator   : {t['operator_iri']}")
    print(f"  Instrument : {device_iri}")
    print(f"  Specimen   : {t['specimen_iri']}")
    print(f"  Results    :")
    for r in c["results"]:
        print(f"    {r['property']:<45}: {r['value']} {r['unit']}")
    return proc_oold["id"], tto_oold["id"], proc_graph, tto_graph


def record_calibration(t, tto_iri):
    cal = t["calibration"]
    oold = _calib_schema.transform({
        "step_name":   cal["name"],
        "model_type":  cal["model_type"],
        "description": cal.get("description", ""),
        "inputs":      [BASE + tto_iri],
        "conditions":  cal.get("optimizer_settings", []),
        "calibrated_parameters": [{"name": p["name"], "value": p["value"],
                                    "unit": p.get("unit", "")}
                                   for p in cal["parameters"]]
    })
    iri = oold["id"]
    graph = _calib_schema.parse(oold, base=BASE)
    print(f"Calibration recorded : {cal['name']}")
    print(f"  Model : {cal['model_type']}")
    for p in cal["parameters"]:
        unit = f" {p['unit']}" if p.get("unit") else ""
        print(f"    {p['name']:<12} = {p['value']}{unit}")
    return iri, graph


def make_material_card(t, calib_iri):
    card = t["material_card"]
    cal  = t["calibration"]
    unit_map = {"MPa": "uqudt:MegaPascal", "GPa": "uqudt:GigaPascal", "%": "uqudt:PERCENT"}
    mech = [{"type": "tto:" + r["property"], "value": r["value"],
              "unit": unit_map.get(r["unit"], r["unit"])}
             for r in t["characterization"]["results"]
             if r["property"] in ("YieldStrength", "TensileStrength",
                                  "PercentageElongationAfterFracture")]
    oold = _card_schema.transform({
        "card_name":             card["name"],
        "material_iri":          t["material_iri"],
        "description":           card.get("description", ""),
        "density":               card["density"],
        "youngs_modulus":        card["youngs_modulus"],
        "poissons_ratio":        card["poissons_ratio"],
        "mechanical_properties": mech,
        "constitutive_model": {
            "model_type":      cal["model_type"],
            "calibration_iri": BASE + calib_iri,
            "parameters":      [{"name": p["name"], "value": p["value"],
                                  "unit": p.get("unit", "")}
                                 for p in cal["parameters"]]
        }
    })
    iri   = oold["id"]
    graph = _card_schema.parse(oold, base=BASE)
    d = card["density"]
    E = card["youngs_modulus"]
    print(f"Material card compiled : {card['name']}")
    print(f"  Density         : {d['value']} {d.get('unit', '')}")
    print(f"  Young's modulus : {E['value']} {E.get('unit', '')}")
    print(f"  Poisson's ratio : {card['poissons_ratio']['value']}")
    print(f"  Flow curve model: {cal['model_type']}")
    return iri, graph


def build_workflow(t, prod_iri, proc_iri, calib_iri):
    c   = t["characterization"]
    cal = t["calibration"]
    p   = t["production"]
    oold = _wf_schema.transform({
        "workflow_name": t["workflow_name"],
        "steps": [
            {"label": p["name"],   "step_type": "pmdco:PMD_0000029",
             "reference": BASE + prod_iri},
            {"label": c["name"],   "step_type": "obi:0000070",
             "reference": BASE + proc_iri},
            {"label": cal["name"], "step_type": "obi:0000471",
             "reference": BASE + calib_iri},
            {"label": "FEM Material Card Export",
             "description": "Export calibrated parameters as solver input files."}
        ]
    })
    graph = _wf_schema.parse(oold, base=BASE)
    print(f"Workflow assembled : {t['workflow_name']}")
    for s in oold["has_part"]:
        print(f"  {s['label']}")
    return graph


# FEM export helpers
_PMD = rdflib.Namespace("https://w3id.org/pmd/co/")
_QDT = rdflib.Namespace("http://qudt.org/schema/qudt/")

def _extract(graph):
    def scalar(iri):
        for node in graph.objects(None, iri):
            v = graph.value(node, _QDT.value)
            if v: return float(v)
    params = {}
    for r in graph.query("""
        PREFIX obi:  <http://purl.obolibrary.org/obo/OBI_>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX qudt: <http://qudt.org/schema/qudt/>
        SELECT ?name ?value WHERE {
          ?c a obi:0000471 ; obi:0000299 ?p .
          ?p rdfs:label ?name ; qudt:value ?value . }
    """):
        params[str(r.name)] = float(r.value)
    return scalar(_PMD["PMD_0000025"]), scalar(_PMD["PMD_0000039"]), scalar(_PMD["PMD_0000040"]), params

def _flow_curve(p, n=21):
    s, s0, c, n_ = p["sigma_sat"], p["sigma_0"], p["c"], p["n"]
    return [(i*0.025, s0 if i==0 else s-(s-s0)*math.exp(-c*(i*0.025)**n_)) for i in range(n)]

def export_fem(graph, out_dir):
    rho, E, nu, params = _extract(graph)
    fc = _flow_curve(params)
    lsd = (f"$$ LS-Dyna *MAT_PIECEWISE_LINEAR_PLASTICITY  (units: t/mm3, MPa)\n"
           f"*MAT_PIECEWISE_LINEAR_PLASTICITY\n"
           f"$# MID        RHO         E           PR\n"
           f"         1  {rho*1e-9:.4e}  {E*1e3:.1f}       {nu:.3f}\n\n"
           f"*DEFINE_CURVE\n$$ plastic strain vs. true stress (MPa)\n         1\n"
           + "\n".join(f"{e:>12.4f}  {s:>12.2f}" for e,s in fc) + "\n\n*END")
    abu = (f"** Abaqus *MATERIAL  (units: kg/m3, Pa)\n"
           f"*MATERIAL, NAME=316L_HS_v1\n*DENSITY\n{rho*1000:.1f},\n"
           f"*ELASTIC\n{E*1e9:.6e}, {nu:.3f}\n"
           f"*PLASTIC\n** yield stress (Pa),  plastic strain\n"
           + "\n".join(f"{s*1e6:>14.2f},  {e:.4f}" for e,s in fc))
    (out_dir / "316L_HS_v1.k").write_text(lsd)
    (out_dir / "316L_HS_v1.inp").write_text(abu)
    print("Solver files saved:")
    print(f"  316L_HS_v1.k    (LS-Dyna)")
    print(f"  316L_HS_v1.inp  (Abaqus)")
    print()
    print(lsd)

print("Ready.")

Ready.


## Step 2: Process the template

Run the cells below one by one. Each one reads a section of your template
and records it. You will see a short summary of what was saved after each cell.

In [4]:
# Register the testing machine
device_iri, device_graph = register_device(workflow_template)

Instrument registered : Zwick Z250 Universal Testing Machine
  Model / S/N         : Z250 / ZR-12345
  Last calibration    : 2024-03-01
  Identifier          : https://example.org/device-zwick-z250-universal-testing-machine


In [5]:
# Record the manufacturing step
prod_iri, prod_graph = record_production(workflow_template)

Production step recorded : 316L Production, batch 1
  Output material      : https://example.org/materials/316L-batch-1
  Rolling Temperature   : 1150 C
  Final Thickness       : 6.0 mm


In [6]:
# Record the tensile test — operator, instrument, and specimen are all required
proc_iri, tto_iri, proc_graph, tto_graph = record_characterization(
    workflow_template, device_iri
)

Tensile test recorded : Tensile test 316L batch 1 QA
  Operator   : https://example.org/people/jane-doe
  Instrument : https://example.org/device-zwick-z250-universal-testing-machine
  Specimen   : https://example.org/specimens/316L-batch-1-bar-1
  Results    :
    YieldStrength                                : 285 MPa
    TensileStrength                              : 620 MPa
    PercentageElongationAfterFracture            : 50 %
    PercentageReductionOfArea                    : 65 %


In [7]:
# Record the constitutive model calibration
calib_iri, calib_graph = record_calibration(workflow_template, tto_iri)

Calibration recorded : Hockett-Sherby calibration 316L batch 1
  Model : Hockett-Sherby
    sigma_sat    = 780.0 MPa
    sigma_0      = 220.0 MPa
    c            = 12.5
    n            = 0.68


In [8]:
# Compile the mechanical material card
card_iri, card_graph = make_material_card(workflow_template, calib_iri)

Material card compiled : 316L stainless steel, Hockett-Sherby v1
  Density         : 7.93 g/cm3
  Young's modulus : 193.0 GPa
  Poisson's ratio : 0.29
  Flow curve model: Hockett-Sherby


In [9]:
# Tie everything together into a named workflow
wf_graph = build_workflow(workflow_template, prod_iri, proc_iri, calib_iri)

Workflow assembled : 316L material card workflow, batch 1
  316L Production, batch 1
  Tensile test 316L batch 1 QA
  Hockett-Sherby calibration 316L batch 1
  FEM Material Card Export


In [10]:
# Merge all records into one connected database
db = rdflib.Graph()
for g in [device_graph, prod_graph, proc_graph, tto_graph, calib_graph, card_graph, wf_graph]:
    db += g
    for prefix, ns in g.namespaces():
        db.bind(prefix, ns)

(HERE / "output_material_card.ttl").write_text(db.serialize(format="turtle"))
print(f"All records connected. Database saved to output_material_card.ttl")

All records connected. Database saved to output_material_card.ttl


## Step 3: Ask questions about your experiment

Because everything is now stored in a structured, connected way, you can
query it like a database. The three cells below ask plain questions and
return plain answers, with no manual searching through files or spreadsheets.

In [11]:
# Question: Who ran the tensile test, and with which instrument?
results = list(db.query("""
    PREFIX obi:    <http://purl.obolibrary.org/obo/OBI_>
    PREFIX prov:   <https://www.w3.org/ns/prov#>
    PREFIX schema: <https://schema.org/>
    PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?test ?operator ?device WHERE {
      ?assay a obi:0000070 ; rdfs:label ?test ;
             prov:wasAssociatedWith ?operator ;
             schema:instrument ?d . ?d rdfs:label ?device . }
"""))

print("Who ran the tensile test?")
print()
for r in results:
    print(f"  Test       : {r.test}")
    print(f"  Operator   : {r.operator}")
    print(f"  Instrument : {r.device}")

Who ran the tensile test?

  Test       : Tensile test 316L batch 1 QA
  Operator   : https://example.org/people/jane-doe
  Instrument : Zwick Z250 Universal Testing Machine


In [12]:
# Question: What were the measured mechanical properties?
results = list(db.query("""
    PREFIX obi: <http://purl.obolibrary.org/obo/OBI_>
    PREFIX tto: <https://w3id.org/pmd/tto/>
    PREFIX iao: <http://purl.obolibrary.org/obo/IAO_>
    PREFIX dcterms: <http://purl.org/dc/terms/>
    SELECT ?property ?value ?unit WHERE {
      ?assay a obi:0000070 ; dcterms:references ?test .
      ?test a tto:TensileTest ; obi:0000299 ?r .
      ?r a ?property ; obi:0001937 ?value ; iao:0000039 ?unit .
    } ORDER BY ?property
"""))

print("What were the measured properties?")
print()
print(f"  {'Property':<45}  {'Value':>8}  Unit")
print(f"  {'-'*45}  {'-'*8}  ----")
for r in results:
    prop = str(r.property).rsplit("/", 1)[-1]
    unit = str(r.unit).rsplit("/", 1)[-1]
    print(f"  {prop:<45}  {float(r.value):>8.1f}  {unit}")

What were the measured properties?

  Property                                          Value  Unit
  ---------------------------------------------  --------  ----
  PercentageElongationAfterFracture                  50.0  PERCENT
  PercentageReductionOfArea                          65.0  PERCENT
  TensileStrength                                   620.0  MegaPascal
  YieldStrength                                     285.0  MegaPascal


In [13]:
# Question: What are the workflow steps and their order?
results = list(db.query("""
    PREFIX bfo:  <http://purl.obolibrary.org/obo/BFO_>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?step ?predecessor WHERE {
      ?wf a bfo:0000015 ; bfo:0000051 ?s . ?s rdfs:label ?step .
      OPTIONAL { ?s bfo:0000062 ?p . ?p rdfs:label ?predecessor . }
    } ORDER BY ?s
"""))

print("What are the workflow steps?")
print()
seen = {}
for r in results:
    label = str(r.step)
    pred  = str(r.predecessor) if r.predecessor else "(start)"
    if label not in seen:
        seen[label] = True
        arrow = "" if pred == "(start)" else f"  (after: {pred})"
        print(f"  {label}{arrow}")

What are the workflow steps?

  316L Production, batch 1
  Tensile test 316L batch 1 QA  (after: 316L Production, batch 1)
  Hockett-Sherby calibration 316L batch 1  (after: Tensile test 316L batch 1 QA)
  FEM Material Card Export  (after: Hockett-Sherby calibration 316L batch 1)


## Step 4: Export your FEM solver files

The material card is compiled from the data you entered in the template.
The flow curve is computed from the Hockett-Sherby parameters.
Two files are saved: one for LS-Dyna, one for Abaqus.

In [14]:
export_fem(db, HERE)

Solver files saved:
  316L_HS_v1.k    (LS-Dyna)
  316L_HS_v1.inp  (Abaqus)

$$ LS-Dyna *MAT_PIECEWISE_LINEAR_PLASTICITY  (units: t/mm3, MPa)
*MAT_PIECEWISE_LINEAR_PLASTICITY
$# MID        RHO         E           PR
         1  7.9300e-09  193000.0       0.290

*DEFINE_CURVE
$$ plastic strain vs. true stress (MPa)
         1
      0.0000        220.00
      0.0250        577.55
      0.0500        670.29
      0.0750        714.61
      0.1000        738.89
      0.1250        753.20
      0.1500        762.06
      0.1750        767.73
      0.2000        771.47
      0.2250        773.98
      0.2500        775.70
      0.2750        776.90
      0.3000        777.74
      0.3250        778.34
      0.3500        778.77
      0.3750        779.08
      0.4000        779.31
      0.4250        779.48
      0.4500        779.61
      0.4750        779.70
      0.5000        779.77

*END


## What just happened?

You filled in one form and ran a handful of cells. Here is what you now have:

| What you got | Where to find it |
|---|---|
| LS-Dyna material card | `316L_HS_v1.k` |
| Abaqus material card | `316L_HS_v1.inp` |
| Connected experiment record | `output_material_card.ttl` |

The experiment record connects every piece of information:

```
Workflow
  Production step  ──────────────────► material batch
  Tensile test ──► who ran it
               ──► which instrument
               ──► which specimen
               ──► measurement results
  Calibration  ──► fitted model + parameters  ──► material card
```

Anyone who picks up the `.ttl` file later (a colleague, an auditor,
a simulation engineer) can answer traceability questions without
digging through lab notebooks or emails:

- Which batch was this material card derived from?
- Who ran the tensile test that fed the calibration?
- Was the testing machine calibrated on the day of the test?
- Which specimen was used?

The FEM files are identical to what you would have produced manually.
The difference is that everything is now documented, connected, and queryable.

---

### What to do next

- **Run it with your own data**: edit the template cell, change the values, re-run.
- **Explore the step-by-step version**: [Notebook 3](../../../../PMDCo/docs/3_material_card_without_template.ipynb)
  shows how each schema call works individually, useful if you want to
  understand or customise the structure.
- **Read the template schema**: [`specs/schema.simplified.json`](../../specs/schema.simplified.json)
  documents every field and its constraints.